# Exercise 4 — Linear, ridge and kernel regression

Today we predict **numbers**, not classes. We start with a line through two
points on the blackboard and end with kernel ridge regression, which fits a
curve without ever building its features.

Everything follows the fourth lecture (linear models and kernels): the closed form
$\mathbf w^{*} = (X^\top X)^{-1} X^\top \mathbf y$, ridge in primal and dual
form, and the kernel matrix $K$.

1. Least squares by hand — two points exactly, three points on paper (Task 1).
2. Linear regression in code, checked against scikit-learn (Task 2).
3. Polynomial features and overfitting; ridge regression and $\lambda$ (Tasks 3–4).
4. Primal and dual form; the kernel trick (Tasks 5–6).
5. Lasso: $L_1$ versus $L_2$ (Task 7).

The parts marked *optional* are extensions for fast students or for home.

In [ ]:
%matplotlib inline
import numpy as np
import numpy.linalg as LA
import matplotlib.pyplot as plt
from ipywidgets import interact, interact_manual
from sklearn.linear_model import LinearRegression, Ridge

np.set_printoptions(precision=4, suppress=True)

## Task 1 — Least squares by hand

### 1a. Two points, one line

With the **bias trick** from the lecture — a column of ones in front of $x$ —
the intercept $b$ becomes just another weight, and two equations become one
matrix equation:

$$\begin{aligned} y_1 &= b + w\,x_1 \\ y_2 &= b + w\,x_2 \end{aligned}
\qquad\Longleftrightarrow\qquad
\mathbf y = X \mathbf w, \quad
X = \begin{bmatrix} 1 & x_1 \\ 1 & x_2 \end{bmatrix}, \quad
\mathbf w = \begin{bmatrix} b \\ w \end{bmatrix}$$

$X$ is square, so $\mathbf w = X^{-1}\mathbf y$. (The lecture writes this
extended vector as $\tilde{\mathbf w}$; we drop the tilde.) The columns are
always ordered $[\,1,\ x,\ x^2, \dots]$, so `w[0]` is the intercept.

In [ ]:
x_two = np.array([1.0, 4.0])
y_two = np.array([2.0, 5.0])
X_two = np.column_stack([np.ones(2), x_two])   # columns [1, x]
print("X =\n", X_two)

# TODO: your code here
w_two = np.zeros(2)
print("w = [b, slope] =", w_two)

x_grid = np.linspace(0, 5, 100)   # x values for drawing lines
plt.plot(x_two, y_two, "o", label="data")
plt.plot(x_grid, w_two[0] + w_two[1] * x_grid, label="fitted line")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

### 1b. Three points — on the blackboard

**Close the laptop for this part.** Three points rarely lie on one line, so
$X\mathbf w = \mathbf y$ has no solution. We minimise the squared error
instead, and that has a closed form:

$$\underset{\mathbf w}{\text{minimize}}\ \lVert X\mathbf w - \mathbf y\rVert_2^2
\qquad\Longrightarrow\qquad
\mathbf w^{*} = (X^\top X)^{-1} X^\top \mathbf y$$

| $x$ | 1 | 2 | 3 |
| --- | - | - | - |
| $y$ | 2 | 3 | 5 |

Compute on paper:

1. $X$, $X^\top X$ and $X^\top \mathbf y$.
2. $(X^\top X)^{-1}$, using $\begin{bmatrix} a & b \\ c & d \end{bmatrix}^{-1} = \frac{1}{ad - bc}\begin{bmatrix} d & -b \\ -c & a \end{bmatrix}$.
3. $\mathbf w^{*}$, the predictions $\hat y_i$, the residuals
   $y_i - \hat y_i$ and their sum of squares.
4. **Ridge** with $\lambda = 1$ on the slope only, as in the lecture's
   appendix "Ridge by hand" (the intercept is not penalised):
   $\mathbf w^{*}_{\text{ridge}} = \left(X^\top X + \begin{bmatrix} 0 & 0 \\ 0 & \lambda \end{bmatrix}\right)^{-1} X^\top \mathbf y$.

What happened to the slope and to the sum of squares? Why is that the point
of $\lambda$?

*Write your calculations and answers here.*

Now let the code confirm your numbers. If a number differs, find out which
of the two is wrong before you go on.

In [ ]:
x_hand = np.array([1.0, 2.0, 3.0])
y_hand = np.array([2.0, 3.0, 5.0])
X_hand = np.column_stack([np.ones(3), x_hand])
penalty = np.diag([0.0, 1.0])   # lambda = 1 on the slope, 0 on the intercept

# TODO: your code here
XtX = np.zeros((2, 2))
Xty = np.zeros(2)
w_ols = np.zeros(2)
w_ridge = np.zeros(2)

print("X^T X =\n", XtX)
print("X^T y =", Xty)
for name, w in [("least squares", w_ols), ("ridge", w_ridge)]:
    residuals = y_hand - X_hand @ w
    print(f"{name:13s}  w = {w}  residuals = {residuals}  "
          f"sum of squares = {np.sum(residuals ** 2):.4f}")

scikit-learn gives the same numbers. It wants $X$ **without** the column of
ones — one row per sample, one column per feature — and adds the intercept
itself. `Ridge` does not penalise the intercept, just like our `penalty`.

In [ ]:
lin = LinearRegression().fit(x_hand.reshape(-1, 1), y_hand)
rid = Ridge(alpha=1.0).fit(x_hand.reshape(-1, 1), y_hand)
print("LinearRegression: b =", lin.intercept_, " slope =", lin.coef_[0])
print("Ridge(alpha=1):   b =", rid.intercept_, " slope =", rid.coef_[0])

plt.plot(x_hand, y_hand, "ko", label="data")
plt.plot(x_grid, w_ols[0] + w_ols[1] * x_grid, label="least squares")
plt.plot(x_grid, w_ridge[0] + w_ridge[1] * x_grid, "--", label="ridge, λ = 1")
plt.vlines(x_hand, y_hand, X_hand @ w_ols, colors="C0", lw=3, label="residuals")
plt.xlim(0, 4)
plt.ylim(0, 6)
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

## Task 2 — Linear regression on many points

With a hundred noisy points only the size of $X$ changes: it is
$100 \times 2$, but $X^\top X$ stays $2 \times 2$. Its size is given by the
number of **features**, not samples — remember this for the dual form in
Task 5.

The data come from $y = 1 + 0.5\,x$ plus Gaussian noise, so we know the
right answer: $\mathbf w = [1,\ 0.5]$.

In [ ]:
np.random.seed(0)
x_lin = 5 * np.random.rand(100)
y_lin = 1 + 0.5 * x_lin + 0.4 * np.random.randn(100)
X_lin = np.column_stack([np.ones(100), x_lin])

# TODO: your code here
w_lin = np.zeros(2)

sk = LinearRegression().fit(x_lin.reshape(-1, 1), y_lin)
print("X:", X_lin.shape, "   X^T X:", (X_lin.T @ X_lin).shape)
print("closed form   :", w_lin)
print("pseudoinverse :", LA.pinv(X_lin) @ y_lin)
print("scikit-learn  :", np.array([sk.intercept_, sk.coef_[0]]))
print("true          :", np.array([1.0, 0.5]))

`LA.pinv` is the pseudoinverse $X^\dagger = (X^\top X)^{-1} X^\top$ from the
lecture. It gets the same result through an SVD, without an explicit inverse,
which matters when $X^\top X$ is close to singular — see the optional part
of Task 4.

### 2b. Can you beat least squares?

Below is a first guess. Set the sliders and click **Run Interact** to try
your own line. Left: the line and its residuals. Right: the MSE of
**every** line $(b, w)$ as a contour map — a bowl with a single bottom. The
star is your closed-form $\mathbf w$. Try to find a line with a lower MSE.

In [ ]:
b_grid, w_grid = np.meshgrid(np.linspace(-1, 3, 81), np.linspace(-0.5, 1.5, 81))
mse_grid = np.mean((b_grid[..., None] + w_grid[..., None] * x_lin - y_lin) ** 2, axis=-1)


def show_line(b=0.0, w=1.0):
    y_hat = b + w * x_lin
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
    ax1.vlines(x_lin, y_lin, y_hat, colors="gray", lw=0.7)
    ax1.plot(x_lin, y_lin, ".")
    ax1.plot(x_grid, b + w * x_grid, "r")
    ax1.set(xlabel="x", ylabel="y", ylim=(-1, 5),
            title=f"your MSE = {np.mean((y_lin - y_hat) ** 2):.3f}")
    ax2.contour(b_grid, w_grid, mse_grid, levels=np.geomspace(0.2, 20, 12), cmap="viridis")
    ax2.plot(w_lin[0], w_lin[1], "k*", ms=14, label="closed form")
    ax2.plot(b, w, "ro", label="your line")
    ax2.set(xlabel="intercept b", ylabel="slope w",
            title=f"closed-form MSE = {np.mean((y_lin - X_lin @ w_lin) ** 2):.3f}")
    ax2.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


show_line(b=0.0, w=1.0)   # a first guess
interact_manual(show_line, b=(-1.0, 3.0, 0.1), w=(-0.5, 1.5, 0.05));

## Task 3 — Polynomial feature transformation

A linear model can fit a curve: it has to be linear in the **weights**, not in
$x$. We keep the model and change only its input. For one input $x$ and a
polynomial of degree $m$, each sample becomes the row
$\Phi(x) = [1,\ x,\ x^2,\ \dots,\ x^m]$, so the feature matrix is

$$X = \begin{bmatrix}
1 & x_1 & x_1^2 & \cdots & x_1^m \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & x_n & x_n^2 & \cdots & x_n^m \end{bmatrix}$$

and the fit is the formula from Tasks 1–2, $\mathbf w = (X^\top X)^{-1} X^\top \mathbf y$.

The data for Tasks 3–6 is a noisy sine: 40 training points, plus 200 test
points from the same process, so that we can tell a good fit from a memorised one.

In [ ]:
np.random.seed(10)
x_train = 5 * np.random.rand(40)
y_train = np.sin(x_train) + 0.3 * np.random.randn(40)
x_test = 5 * np.random.rand(200)
y_test = np.sin(x_test) + 0.3 * np.random.randn(200)

plt.plot(x_train, y_train, ".", label="train (40)")
plt.plot(x_grid, np.sin(x_grid), "k--", lw=1, label="true sin(x)")
plt.ylim(-2, 2)
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

### 3a. The feature matrix, by hand

First we map $x$ from $[0, 5]$ to $t \in [-1, 1]$ and take powers of $t$.
The model does not change (a polynomial in $x$ is also a polynomial in $t$),
but all powers stay between −1 and 1 instead of reaching
$5^{15} \approx 3 \cdot 10^{10}$. Without this, `inv` returns numerical
garbage at degree 15 (see the optional part at the end of Task 4).

**On paper:** for $x = 0,\ 2.5,\ 5$ and degree 3, write $t$ and the
$3 \times 4$ matrix $X$. Then complete `poly_features` (column $j$ is
`t ** j`; `np.column_stack(list_of_columns)` glues them together) and compare
it with your matrix and with scikit-learn's `PolynomialFeatures` (which also
handles several inputs: for $[x_1, x_2]$ and degree 2 it gives
$[1,\ x_1,\ x_2,\ x_1^2,\ x_1 x_2,\ x_2^2]$).

In [ ]:
from sklearn.preprocessing import PolynomialFeatures


def scale_x(x):
    # map the input range [0, 5] to [-1, 1]
    return 2 * x / 5 - 1


def poly_features(x, degree):
    # rows [1, t, t^2, ..., t^degree] with t = scale_x(x)
    t = scale_x(x)
    # TODO: your code here
    return np.ones((len(x), degree + 1))


x_small = np.array([0.0, 2.5, 5.0])
print(poly_features(x_small, 3))
print(PolynomialFeatures(3).fit_transform(scale_x(x_small).reshape(-1, 1)))

### 3b. Too simple, about right, too flexible

Fit degrees 1, 3 and 15: inside the loop, complete the one line that computes
$\mathbf w$ (the least-squares formula from Task 2). The loop prints the
training and test error (MSE) and the size of the weights $\lVert\mathbf w\rVert_2$.

In [ ]:
def mse(y, y_pred):
    return np.mean((y - y_pred) ** 2)


fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, degree in zip(axes, [1, 3, 15]):
    X = poly_features(x_train, degree)
    # TODO: your code here
    w = np.zeros(degree + 1)
    test_mse = mse(y_test, poly_features(x_test, degree) @ w)
    print(f"degree {degree:2d}: train MSE {mse(y_train, X @ w):.4f}, "
          f"test MSE {test_mse:7.4f}, ||w|| {LA.norm(w):8.1f}")

    ax.plot(x_train, y_train, ".")
    ax.plot(x_grid, poly_features(x_grid, degree) @ w, "r")
    ax.set_title(f"degree {degree}, test MSE {test_mse:.3f}")
    ax.set_xlabel("x")
axes[0].set_ylim(-2, 2)
axes[0].set_ylabel("y")
plt.tight_layout()
plt.show()

Degree 1 is too rigid (**underfitting**), degree 3 is about right, and
degree 15 wiggles through the training points (**overfitting**): the smallest
training error of the three, but the largest test error by far. Its weights are
huge — the wiggles are large coefficients that almost cancel each other. Ridge
regression penalises exactly that.

## Task 4 — Ridge regression: $\lambda$ in numbers, then in a picture

Choosing the degree is guesswork. Ridge regression keeps the high degree but
penalises large weights:

$$\underset{\mathbf w}{\text{minimize}}\
\lVert X\mathbf w - \mathbf y\rVert_2^2 + \lambda \lVert \mathbf w \rVert_2^2
\qquad\Longrightarrow\qquad
\mathbf w^{*} = (X^\top X + \lambda I)^{-1} X^\top \mathbf y$$

(The gradient $-2X^\top \mathbf y + 2(X^\top X + \lambda I)\mathbf w$ set to
zero gives the formula.) With $\lambda = 0$ it is the least-squares fit of
Task 3.

Implement `ridge_fit` and check it against scikit-learn's `Ridge` (its
`alpha` is our $\lambda$).

**Intercept — a difference from Task 1b.** There we penalised the slope only.
Here we use the plain formula with $\lambda I$, which penalises **every**
weight, the intercept `w[0]` included. It is simpler to write, and for the
small $\lambda$ that work well below it changes the test error by less than
$10^{-4}$; only a very large $\lambda$ shows the difference (the last row of
the table). `Ridge` computes the same when it gets our $X$ with the ones
column and `fit_intercept=False`.

In [ ]:
def ridge_fit(X, y, lam):
    # TODO: your code here
    return np.zeros(X.shape[1])


X15_train = poly_features(x_train, 15)
X15_test = poly_features(x_test, 15)

w = ridge_fit(X15_train, y_train, 0.01)
w_sklearn = Ridge(alpha=0.01, fit_intercept=False).fit(X15_train, y_train).coef_
print("largest difference from sklearn Ridge:", np.max(np.abs(w - w_sklearn)))

**$\lambda$ in numbers.** Before running the next cell, guess: as $\lambda$
grows from 0 to $10^4$, what happens to $\lVert\mathbf w\rVert_2$, to the
training error and to the test error? Then run it and answer:

1. Which $\lambda$ gives the smallest test error?
2. From $\lambda = 0$ to $\lambda = 10^{-4}$: how much does the training error
   change, and how much do $\lVert\mathbf w\rVert_2$ and the test error change?
3. What is the model at $\lambda = 10^4$?

(In a real project, $\lambda$ is chosen on a validation set, not on the test
set — lab 2.)

In [ ]:
print(f"{'lambda':>8} {'||w||':>10} {'train MSE':>10} {'test MSE':>10}")
for lam in [0, 1e-6, 1e-4, 1e-2, 1, 1e2, 1e4]:
    w = ridge_fit(X15_train, y_train, lam)
    print(f"{lam:8.0e} {LA.norm(w):10.3g} "
          f"{mse(y_train, X15_train @ w):10.4f} {mse(y_test, X15_test @ w):10.4f}")

*Write your three answers here.*

**$\lambda$ in a picture.** The same model for four rows of the table.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for ax, lam in zip(axes, [0, 1e-4, 1, 1e4]):
    w = ridge_fit(X15_train, y_train, lam)
    ax.plot(x_train, y_train, ".")
    ax.plot(x_grid, poly_features(x_grid, 15) @ w, "r")
    ax.set_title(f"$\\lambda$ = {lam:g}, test MSE {mse(y_test, X15_test @ w):.3f}")
    ax.set_xlabel("x")
axes[0].set_ylim(-2, 2)
axes[0].set_ylabel("y")
plt.tight_layout()
plt.show()

### Optional — explore degree and $\lambda$ together

Run the cell, set the two sliders and click **Run Interact** (Colab or
Jupyter). Find a combination that overfits, one that underfits, and a
degree-15 model that is as good as degree 3. The title shows both errors.

In [ ]:
#@title Degree and λ sliders (run this cell)
def show_fit(degree=15, log10_lambda=-4):
    lam = 10.0 ** log10_lambda
    w = ridge_fit(poly_features(x_train, degree), y_train, lam)
    train_mse = mse(y_train, poly_features(x_train, degree) @ w)
    test_mse = mse(y_test, poly_features(x_test, degree) @ w)
    plt.figure(figsize=(7, 3.6))
    plt.plot(x_train, y_train, ".", label="train")
    plt.plot(x_grid, np.sin(x_grid), "k--", lw=1, label="true sin(x)")
    plt.plot(x_grid, poly_features(x_grid, degree) @ w, "r", label="fit")
    plt.ylim(-2, 2)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"degree {degree}, $\\lambda$ = 1e{log10_lambda}: "
              f"train MSE {train_mse:.3f}, test MSE {test_mse:.3f}")
    plt.legend(loc="lower left")
    plt.show()


show_fit(degree=15, log10_lambda=-4)   # a first setting
interact_manual(show_fit, degree=(1, 15), log10_lambda=(-8, 4));

### Optional — why we scaled $x$

The condition number $\operatorname{cond}(A)$ measures how much a matrix
inverse amplifies rounding errors; `float64` keeps about 16 significant digits,
so beyond roughly $10^{16}$ the inverse is noise. Compare $X^\top X$ built from
the raw $x \in [0, 5]$, from the scaled $t \in [-1, 1]$, and the ridge matrix
$X^\top X + \lambda I$. The last line fits degree 15 on the **raw** $x$;
compare its training error with Task 3.

In [ ]:
for degree in [1, 3, 9, 15]:
    X_raw = np.column_stack([x_train ** j for j in range(degree + 1)])
    X_sc = poly_features(x_train, degree)
    I = np.eye(degree + 1)
    print(f"degree {degree:2d}: cond raw {LA.cond(X_raw.T @ X_raw):8.1e}, "
          f"scaled {LA.cond(X_sc.T @ X_sc):8.1e}, "
          f"scaled + 1e-4 I {LA.cond(X_sc.T @ X_sc + 1e-4 * I):8.1e}")

w_raw = LA.inv(X_raw.T @ X_raw) @ X_raw.T @ y_train
print("degree 15 on raw x: train MSE", mse(y_train, X_raw @ w_raw))

## Task 5 — Primal and dual form

The ridge solution of Task 4 inverts $X^\top X + \lambda I$, a $d \times d$
matrix — one row per **feature**. The same solution can be written with
$X X^\top + \lambda I$, an $n \times n$ matrix — one row per **sample**:

$$\underbrace{\mathbf w^{*} = (X^\top X + \lambda I)^{-1} X^\top \mathbf y}_{\text{primal: } d \times d}
\qquad\qquad
\underbrace{\boldsymbol\alpha^{*} = (X X^\top + \lambda I)^{-1} \mathbf y,
\quad \mathbf w = X^\top \boldsymbol\alpha}_{\text{dual: } n \times n}$$

The dual learns one coefficient $\alpha_i$ per **training sample**, and
$\mathbf w = X^\top\boldsymbol\alpha = \sum_i \alpha_i \mathbf x_i$ is a
combination of the training samples. A new sample is predicted without
forming $\mathbf w$ at all:
$\hat y_{\text{new}} = \sum_i \alpha_i\, \mathbf x_i^\top \mathbf x_{\text{new}}$.

* Many samples, few features $\to$ the primal inverts the smaller matrix.
* Few samples, very many features $\to$ the dual does.
* In the dual, the samples appear **only inside dot products**
  $\mathbf x_i^\top \mathbf x_j$ — Task 6 is built on exactly this.

### 5a. Two points — on the blackboard

One feature, **no** column of ones, $\lambda = 1$:

| $x$ | 1 | 2 |
| --- | - | - |
| $y$ | 1 | 3 |

1. **Primal.** $X^\top X$ is $1 \times 1$, so the inverse is a division:
   $w = \sum_i x_i y_i \,/\, (\sum_i x_i^2 + \lambda)$.
2. **Dual.** Write $X X^\top + \lambda I$ ($2 \times 2$), invert it as in
   Task 1, compute $\boldsymbol\alpha$ and then $w = X^\top\boldsymbol\alpha$.
3. Predict $y$ at $x_{\text{new}} = 3$ both ways: $w \cdot 3$ and
   $\sum_i \alpha_i\, x_i \cdot 3$.

How large would the two matrices be for 1000 samples with 10 features?

*Write your calculations and answers here.*

Now let the code confirm the blackboard. The primal is `ridge_fit` from
Task 4; write the dual.

In [ ]:
X_small = np.array([[1.0], [2.0]])   # n = 2 samples (rows), d = 1 feature
y_small = np.array([1.0, 3.0])
lam = 1.0

w_primal = ridge_fit(X_small, y_small, lam)      # the primal, from Task 4
# TODO: your code here
alpha_small = np.zeros(2)
w_dual = np.zeros(1)

x_new = np.array([[3.0]])
print("primal: inverts", (X_small.T @ X_small).shape, "  w =", w_primal)
print("dual:   inverts", (X_small @ X_small.T).shape, "  alpha =", alpha_small)
print("        w = X^T alpha =", w_dual)
print("prediction at x = 3:  primal", x_new @ w_primal,
      "  dual", x_new @ X_small.T @ alpha_small)

### 5b. The same on the polynomial features

The noisy sine from Task 3 with degree-10 features: $n = 40$ samples,
$d = 11$ features. The primal is `ridge_fit` again; write the dual and
predict the test set with
$\hat{\mathbf y}_{\text{test}} = X_{\text{test}} X^\top \boldsymbol\alpha$
— no $\mathbf w$ anywhere.

In [ ]:
degree, lam = 10, 1e-2
X_p = poly_features(x_train, degree)        # 40 x 11
X_p_test = poly_features(x_test, degree)    # 200 x 11

w_p = ridge_fit(X_p, y_train, lam)          # the primal
y_test_primal = X_p_test @ w_p
# TODO: your code here
alpha_p = np.zeros(len(x_train))
y_test_dual = np.zeros(len(x_test))

print("primal inverts", (X_p.T @ X_p).shape, "   dual inverts", (X_p @ X_p.T).shape)
print("test MSE: primal", round(mse(y_test, y_test_primal), 4),
      "  dual", round(mse(y_test, y_test_dual), 4))
print("largest difference of the predictions:", np.max(np.abs(y_test_primal - y_test_dual)))

## Task 6 — The kernel trick

The dual touches the data only through $X X^\top$ (dot products between
training samples) and $X_{\text{new}} X^\top$ (new against training samples).
Replace every dot product with a **kernel**

$$k(\mathbf x_i, \mathbf x_j) = \Phi(\mathbf x_i)^\top \Phi(\mathbf x_j)$$

and the dual works in the feature space of $\Phi$ without ever computing
$\Phi$. The **kernel matrix** $K_{ij} = k(\mathbf x_i, \mathbf x_j)$ takes the
place of $X X^\top$, and ridge regression becomes **kernel ridge regression**
(KRR):

$$\boldsymbol\alpha^{*} = (K + \lambda I)^{-1}\mathbf y,
\qquad
\hat{\mathbf y}_{\text{new}} = K_{\text{new}}\,\boldsymbol\alpha,
\quad (K_{\text{new}})_{ij} = k(\mathbf x_{\text{new},i}, \mathbf x_j)$$

### 6a. One number, two routes

The example from the lecture: $\Phi(\mathbf x) = (x_1^2,\ x_2^2,\ \sqrt2\,x_1 x_2)$,
$\mathbf x_i = (1, 2)$, $\mathbf x_j = (3, 1)$.

In [ ]:
x_i = np.array([1.0, 2.0])
x_j = np.array([3.0, 1.0])

def phi(x):
    return np.array([x[0] ** 2, x[1] ** 2, np.sqrt(2) * x[0] * x[1]])

print("explicit:  Phi(x_i)^T Phi(x_j) =", phi(x_i) @ phi(x_j))
print("kernel:    (x_i^T x_j)^2       =", (x_i @ x_j) ** 2)

Both give 25: the first after building a 3-dimensional image of each sample,
the second from one dot product in the original 2 dimensions. For
100-dimensional inputs and degree 3, the explicit route needs 176 851 numbers
per sample; the kernel still needs one dot product.

### 6b. Three kernels

$$k_{\text{lin}}(\mathbf x_i, \mathbf x_j) = \mathbf x_i^\top \mathbf x_j,
\qquad
k_{\text{poly}}(\mathbf x_i, \mathbf x_j) = (\mathbf x_i^\top \mathbf x_j + c)^p,
\qquad
k_{\text{rbf}}(\mathbf x_i, \mathbf x_j) = \exp\!\big(-\gamma \lVert \mathbf x_i - \mathbf x_j\rVert^2\big)$$

with $\gamma = 1/(2\sigma^2)$. The polynomial kernel belongs to a polynomial
feature space like the one in Task 3; the RBF kernel belongs to an
**infinite-dimensional** $\Phi$, which could never be built explicitly.

Each function takes two sample matrices, $A$ ($m \times d$) and $B$
($n \times d$), and returns the $m \times n$ matrix of all kernel values.
The linear kernel and the squared distances are given.

In [ ]:
def kernel_linear(A, B):
    return A @ B.T


def kernel_poly(A, B, p=3, c=1.0):
    # TODO: your code here
    return np.zeros((A.shape[0], B.shape[0]))


def kernel_rbf(A, B, gamma=1.0):
    sq_dist = np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=2)   # m x n
    # TODO: your code here
    return np.zeros((A.shape[0], B.shape[0]))


# compare with scikit-learn on random data
from sklearn.metrics.pairwise import polynomial_kernel, rbf_kernel

A = np.random.default_rng(0).random((5, 3))
B = np.random.default_rng(1).random((4, 3))
print("polynomial:", np.allclose(kernel_poly(A, B), polynomial_kernel(A, B, degree=3, gamma=1, coef0=1)))
print("RBF       :", np.allclose(kernel_rbf(A, B, gamma=0.7), rbf_kernel(A, B, gamma=0.7)))

### 6c. Kernel ridge regression

KRR is the dual of Task 5 with $X X^\top$ replaced by $K$. Write the fit —
one line. The prediction $K_{\text{new}}\boldsymbol\alpha$ is then a matrix
product.

In [ ]:
def krr_fit(K, y, lam):
    # dual coefficients: alpha = (K + lam I)^-1 y
    # TODO: your code here
    return np.zeros(len(y))


X_tr = x_train.reshape(-1, 1)        # kernels take sample matrices (n x d)
X_te = x_test.reshape(-1, 1)
X_plot = np.linspace(0, 5, 200).reshape(-1, 1)

K = kernel_rbf(X_tr, X_tr, gamma=1.0)
alpha_rbf = krr_fit(K, y_train, lam=0.1)
y_test_krr = kernel_rbf(X_te, X_tr, gamma=1.0) @ alpha_rbf

print("K:", K.shape, " K_test:", kernel_rbf(X_te, X_tr).shape, " alpha:", alpha_rbf.shape)
print("RBF kernel ridge, test MSE:", round(mse(y_test, y_test_krr), 4))

Two checks, because a plausible-looking number proves nothing:

1. KRR with the **linear kernel on the polynomial features** of Task 5 must
   give exactly the dual of Task 5: $K = X X^\top$.
2. Our RBF KRR must match `sklearn.kernel_ridge.KernelRidge`, whose `alpha`
   is our $\lambda$.

In [ ]:
from sklearn.kernel_ridge import KernelRidge

alpha_lin = krr_fit(kernel_linear(X_p, X_p), y_train, lam=1e-2)
print("linear kernel = dual of Task 5:",
      np.allclose(kernel_linear(X_p_test, X_p) @ alpha_lin, y_test_dual))

sk_krr = KernelRidge(alpha=0.1, kernel="rbf", gamma=1.0).fit(X_tr, y_train)
print("RBF KRR = sklearn KernelRidge: ", np.allclose(y_test_krr, sk_krr.predict(X_te)))

**Which kernel?** The same data, the same $\lambda = 0.1$, three kernels on
the raw input $x$.

In [ ]:
plt.plot(x_train, y_train, "k.", label="train")
for name, kernel in [("linear", kernel_linear),
                     ("polynomial, p = 3", kernel_poly),
                     ("RBF, gamma = 1", kernel_rbf)]:
    alpha_k = krr_fit(kernel(X_tr, X_tr), y_train, lam=0.1)
    test_mse = mse(y_test, kernel(X_te, X_tr) @ alpha_k)
    plt.plot(X_plot, kernel(X_plot, X_tr) @ alpha_k, label=f"{name}  (test MSE {test_mse:.3f})")
plt.ylim([-2, 2])
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

### 6d. What the RBF model looks like

With the RBF kernel, the prediction
$\hat y(\mathbf x) = \sum_i \alpha_i\, k(\mathbf x_i, \mathbf x)$ is a **sum
of Gaussian bumps**: one centred at every training sample, with height
$\alpha_i$ and width set by $\gamma$. The thin lines below are the individual
bumps, the red line is their sum. Left: only the first six training samples,
so that each bump can be seen; right: all 40.

In [ ]:
def plot_krr(ax, n, gamma, lam, bumps=True):
    # RBF kernel ridge fitted on the first n training samples
    X, y = X_tr[:n], y_train[:n]
    alpha = krr_fit(kernel_rbf(X, X, gamma), y, lam)
    K_plot = kernel_rbf(X_plot, X, gamma)
    if bumps:
        ax.plot(X_plot, K_plot * alpha, color="C0", lw=0.8)   # alpha_i k(x_i, x)
    ax.plot(X_plot, K_plot @ alpha, "r", lw=2)
    ax.plot(X, y, "k.")
    test_mse = mse(y_test, kernel_rbf(X_te, X, gamma) @ alpha)
    ax.set_title(f"n = {n}, $\\gamma$ = {gamma:.3g}, $\\lambda$ = {lam:.3g}\n"
                 f"train MSE {mse(y, kernel_rbf(X, X, gamma) @ alpha):.3f}, "
                 f"test MSE {test_mse:.3f}", fontsize=9)


fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_krr(axes[0], n=6, gamma=1.0, lam=0.1)
plot_krr(axes[1], n=40, gamma=1.0, lam=0.1)
for ax in axes:
    ax.set_xlabel("x")
plt.tight_layout()
plt.show()
print("alpha of the six-sample model:",
      krr_fit(kernel_rbf(X_tr[:6], X_tr[:6]), y_train[:6], 0.1).round(2))

With 40 overlapping bumps the individual $\alpha_i$ are large, and
neighbouring bumps nearly cancel — the same thing the large polynomial weights
did in Task 3. $\lambda$ limits the size of $\boldsymbol\alpha$, $\gamma$ sets
how narrow the bumps are.

### 6e. $\gamma$ and $\lambda$

The two hyperparameters pull in opposite directions. Each panel below is one
combination; read the train and test errors in the titles before you look at
the curves.

In [ ]:
gammas = [0.03, 0.3, 3.0, 30.0]
lams = [1e-3, 1e-1, 1e1]

fig, axes = plt.subplots(len(gammas), len(lams), figsize=(12, 11), sharex=True, sharey=True)
for r, gamma in enumerate(gammas):
    for c, lam in enumerate(lams):
        plot_krr(axes[r, c], n=40, gamma=gamma, lam=lam, bumps=False)
axes[0, 0].set_ylim([-2, 2])
for ax in axes[-1]:
    ax.set_xlabel("x")
plt.tight_layout()
plt.show()

*Which panel has the smallest test error? What happens for a large $\gamma$
with a small $\lambda$, and why?*

### Optional — move the sliders

Run the cell, set the sliders and click **Run Interact** (Colab or Jupyter):
the number of training samples $n$, $\log_{10}\gamma$ and $\log_{10}\lambda$. Try narrow bumps
($\log_{10}\gamma = 1.5$) with a tiny $\lambda$, then increase $\lambda$.

In [ ]:
#@title Kernel ridge sliders (run this cell)
def show_krr(n=6, log10_gamma=0.0, log10_lambda=-1.0):
    fig, ax = plt.subplots(figsize=(7, 4))
    plot_krr(ax, n, 10 ** log10_gamma, 10 ** log10_lambda)
    ax.set_xlabel("x")
    plt.show()


show_krr(n=6, log10_gamma=1.5, log10_lambda=-4.0)   # narrow bumps, tiny lambda
interact_manual(show_krr, n=(2, 40), log10_gamma=(-2.0, 2.0, 0.1), log10_lambda=(-4.0, 2.0, 0.1));

## Task 7 — Lasso: $L_1$ versus $L_2$

Ridge penalises the squared $L_2$ norm of the weights. **Lasso** penalises
their $L_1$ norm (fourth lecture):

$$\underset{\mathbf w}{\text{minimize}}\
\lVert X\mathbf w - \mathbf y\rVert_2^2 + \lambda \lVert \mathbf w \rVert_1,
\qquad
\lVert \mathbf w \rVert_1 = \sum_j |w_j|$$

**Why bother:** $L_2$ shrinks small weights but never makes them zero; $L_1$
sets them **exactly to zero**. Lasso therefore selects features while it
fits — the *embedded* method from the third lecture. There is no closed form
($|w|$ has a kink at zero), so we use scikit-learn's solver.

The data: the diabetes dataset from scikit-learn — 442 patients, 10 clinical
features, and the target is the disease progression one year later. The
features are standardised, so that $\lambda$ penalises all of them on the
same scale.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

diabetes = load_diabetes()
names = diabetes.feature_names
Xd_train, Xd_valid, yd_train, yd_valid = train_test_split(    # d = diabetes
    diabetes.data, diabetes.target, test_size=0.25, random_state=42)

scaler = StandardScaler().fit(Xd_train)          # fitted on the training part only
Xd_train, Xd_valid = scaler.transform(Xd_train), scaler.transform(Xd_valid)

print(names)
print("train / validation:", len(yd_train), len(yd_valid))

Numbers first. For every $\lambda$, fit `Ridge` and `Lasso` on the training
part and print the **validation MSE** and the number of **non-zero** weights.

In scikit-learn, $\lambda$ is called `alpha`. `Lasso` divides the squared
error by $2n$, so the same `alpha` is a much stronger penalty for Lasso than
for Ridge — compare the trends in each column, not the two models row by row.

In [ ]:
lambdas = [0.01, 0.1, 1, 10, 100, 1000]

print(f"{'lambda':>8} | {'Ridge MSE':>9} {'non-zero':>8} | {'Lasso MSE':>9} {'non-zero':>8}")
for lam in lambdas:
    row = f"{lam:8g} |"
    for Model in [Ridge, Lasso]:
        # fit Model(alpha=lam) on the training part; mse() is from Task 3
        # TODO: your code here
        valid_mse = 0.0   # MSE on the validation part
        n_nonzero = 0     # number of non-zero weights in model.coef_
        row += f" {valid_mse:9.1f} {n_nonzero:8d} |"
    print(row)

The same table as a picture: every line is one weight as $\lambda$ grows.

In [ ]:
lambdas_path = np.logspace(-2, 3, 50)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, Model in zip(axes, [Ridge, Lasso]):
    weights = [Model(alpha=lam).fit(Xd_train, yd_train).coef_ for lam in lambdas_path]
    ax.plot(lambdas_path, weights)                # one line per feature
    ax.axhline(0, color="k", lw=0.8)
    ax.set(xscale="log", xlabel="$\\lambda$ (alpha)", title=Model.__name__)
axes[0].set_ylabel("weight")
axes[1].legend(names, fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

*Which $\lambda$ gives the lowest validation MSE for each model, and how many
features does Lasso keep there? Which features survive the longest in the
Lasso path?*

**Why exactly zero?** Take a single weight whose least-squares value is
$a = 2$ and add each penalty. $w^2$ is flat at zero, so the minimum of
$(w-a)^2 + \lambda w^2$ only approaches zero. $|w|$ has a kink at zero, and
once $\lambda \ge 2a$ the minimum of $(w-a)^2 + \lambda |w|$ sits exactly on
the kink. Run the cell and move the slider.

In [ ]:
#@title Why does L1 stop at exactly zero? (run this cell)
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

a = 2.0                                    # the least-squares value of one weight
w_axis = np.linspace(-1, 3, 801)             # candidate values of the weight
lambdas_1d = np.linspace(0, 6, 16)
penalties = {"$L_2$: $(w-a)^2+\\lambda w^2$": w_axis ** 2,     # ridge, one weight
             "$L_1$: $(w-a)^2+\\lambda |w|$": np.abs(w_axis)}  # lasso, one weight
w_best = {name: [w_axis[np.argmin((w_axis - a) ** 2 + lam * p)] for lam in lambdas_1d]
          for name, p in penalties.items()}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), dpi=72)
def draw(i):
    lam = lambdas_1d[i]
    for ax, color, (name, p) in zip(axes, ["C0", "C1"], penalties.items()):
        ax.clear()
        ax.plot(w_axis, (w_axis - a) ** 2 + lam * p, color)
        ax.plot(w_best[name][i], np.min((w_axis - a) ** 2 + lam * p), "ro")
        ax.axvline(0, color="k", lw=0.8)
        ax.set(title=name, xlabel="$w$", ylim=(0, 12))
    axes[2].clear()
    for name in penalties:
        axes[2].plot(lambdas_1d, w_best[name], label=name.split(":")[0])
        axes[2].plot(lam, w_best[name][i], "ro")
    axes[2].set(title=f"$\\lambda$ = {lam:.1f}", xlabel="$\\lambda$", ylabel="best $w$")
    axes[2].legend()
    fig.tight_layout()

animation = FuncAnimation(fig, draw, frames=len(lambdas_1d))
plt.close(fig)
HTML(animation.to_jshtml(fps=4, default_mode="reflect"))

## Summary

- Linear regression is one formula,
  $\mathbf w = (X^\top X)^{-1}X^\top \mathbf y$ — on paper for three points,
  in NumPy for a hundred.
- A feature transformation curves the fit, but the model stays linear in
  $\mathbf w$; a degree-15 polynomial on 40 points overfits.
- Ridge adds $\lambda I$ to $X^\top X$: smaller weights, less overfitting.
  $\lambda$ is chosen on validation data.
- The dual form touches the samples only through dot products; replacing them
  by a kernel gives kernel ridge regression. With the RBF kernel it is a sum
  of bumps, their width set by $\gamma$.
- $L_2$ shrinks all weights; $L_1$ sets some of them exactly to zero and so
  selects features.

**Two questions:** Why does ridge help the degree-15 polynomial? Why does
Lasso give exact zeros while ridge does not?